# Chain

**Chain**(체인)은 여러 컴포넌트(요소)를 정해진 순서대로 연결하여 **복잡한 AI 작업을 단계별로 자동화**할 수 있도록 돕는 구조이다.

- 각 컴포넌트는 **이전 처리결과를 입력으로 받아 처리한 후 다음 단계로 결과를 전달**한다.
- 복잡한 작업을 여러 개의 단순한 단계로 나누고, 각 단계를 순차적으로 실행함으로써 전체 작업을 체계적으로 구성할 수 있다.

## 기본 개념

- 체인은 하나의 LLM 호출에 그치지 않고 **여러 LLM 호출이나 도구 실행등을 순차적으로 연결**하여 실행 할 수 있다.
- 예를 들어, 사용자의 질문 → 검색 → 요약 → 응답 생성 같은 일련의 작업을 체인으로 구성할 수 있다.
- 이러한 체인구조를 사용하면 **작업흐름이 명확**해지고 **코드의 재사용성**이 높아지며 **유지 보수 및 확장성이** 향상된다.

## LangChain에서의 Chain 구성 방식

LangChain은 다음 두 가지 방식을 통해 체인을 구성할 수 있다.

### 1. Off-the-shelf Chains 방식 (클래식 방식)

- LangChain에서 제공하는 **미리 정의된 Chain 클래스**(예: `LLMChain`, `SequentialChain`, `SimpleSequentialChain`)를 활용하는 방식이다.
- 각 클래스는 다양한 chain 알고리즘들을 미리 구현한 것으로 상황에 맞는 것을 선택하여 필요한 구성요소를 전달해 생생한다.
- 이 방식은 LangChain의 **초기 방식**이며, 새로운 기능 확장이나 유연한 구성에 한계가 있기 때문에 현재 **더 이상 사용되지 않음(deprecated)** 상태이다.
  - 현재 LangChain에서는 권장하지 않는 방식이다.

### 2. LCEL (LangChain Expression Language) 방식

- LCEL은 체인을 표현식(Expression) 기반의 선언적 파이프라인 방식으로 구성할 수 있도록 설계된 최신 체인 구성 방법이다. 
- 각 컴포넌트들을 `|` 연산자로 연결하여, 흐름이 자연스럽게 이어지는 형태의 체인을 구성한다.
- LCEL 방식은 간결하고 선언적인 문법을 제공하여 **직관적이고 융통성과 확장성 있는 체인 구성**이 가능하다.
- LCEL은
  - 선형적 흐름 구조를 가진다.
  - 문법이 간결하고 선언적이다.
  - 체인의 구조가 코드만 봐도 쉽게 파악된다.
  - 유연하고 확장성이 매우 뛰어나다.
- `Runnable` 기반 구조
  - LCEL방식을 구성하는 모든 컴포넌트들은 `Runnable` 이라는 공통 인터페이스를 기반으로 동작한다.
  - 체인을 구성하는 각 컴포넌트들은 `Runnable` 을 상속하여 구현하여 이를 통해 일관된 실행 인터페이스를 제공한다.
  - **공통 메소드**:
    - `invoke()`: 단일 입력에 대한 처리
    - `batch()`: 다수 입력을 묶어서 한번에 처리
    - `stream()`: 스트리밍 방식의 요청
    - `ainvoke()`, `abatch()`, `astream()`: 비동기적 처리 메소드

In [2]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
prompt = ChatPromptTemplate.from_template(
    template="{item}에 어울리는 브랜드 이름 {count}개를 만들어 주세요."
)
model = ChatOpenAI(model="gpt-5.4-nano")
parser = StrOutputParser()

In [3]:
############################################################
# 기존의 Off the shell 방식 - langchain-classic 설치 필요
############################################################
from langchain_classic import LLMChain
# chain을 구성하는 요소들을 넣어서 생성.
# prompt_template-[prompt]->model-[응답]->output parser->최종결과
chain = LLMChain(
    prompt=prompt,
    llm=model,
    output_parser=parser
)

res = chain.invoke({"item":"가방", "count":3})
print(res)


C:\Users\Playdata\AppData\Local\Temp\ipykernel_2756\1380311513.py:7: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(


{'item': '가방', 'count': 3, 'text': '물론이죠! 가방에 어울리는 브랜드 이름 3개 제안드릴게요.\n\n1. **루미바울**\n2. **노블에코백**\n3. **마일드스테치**'}


In [4]:
##############################
#  LCEL
##############################
chain2 = prompt | model | parser
print(type(chain2))
res2 = chain2.invoke({"item":"TV 브랜드", "count":3}) 

<class 'langchain_core.runnables.base.RunnableSequence'>


In [5]:
res2

'1. **리바인TV**  \n2. **루미브릭**  \n3. **네오비전스**'

In [ ]:
from langchain_core.runnables import Runnable
isinstance(model, Runnable), isinstance(prompt, Runnable), isinstance(parser, Runnable), isinstance(chain2, Runnable)   # Runnable 타입인지 확인

(True, True, True, True)

# Runnable 타입 주요 클래스


## [Runnable](https://reference.langchain.com/python/langchain_core/runnables/#langchain_core.runnables.base.Runnable)
- LangChain의 Runnable은 실행 가능한 작업 단위를 캡슐화한 개념으로, 데이터 흐름의 각 단계를 정의하고 **체인(chain) 에 포함 되어**  복잡한 작업의 각 단계를 수행 한다.
- **Chain을 구성하는 class들**은 Runnable의 상속 받아 구현한다.
- **Prompt Template클래스**, **Chat 모델**, **Output Parser 클래스** 등 다양한 컴포넌트가 Runnable을 상속받아 구현된다.

### 주요 특징
- 작업 단위의 캡슐화:
    - Runnable은 특정 작업(예: 프롬프트 생성, LLM 호출, 출력 파싱 등)을 수행하는 독립적인 컴포넌트이다.
    - 각 컴포넌트는 독립적으로 테스트 및 재사용이 가능하며, 조합하여 복잡한 체인을 구성할 수 있다.
- 체인 연결 및 작업 흐름 관리:
    - Runnable은 체인(chain, 일련의 연결된 작업 흐름)을 구성하는 기본 단위로 사용된다.
    - LangChain Expression Language(LCEL)를 사용하면 | 연산자를 통해 여러 Runnable을 쉽게 연결할 수 있다.
    - 입력과 출력의 형식을 일관되게 유지하여 각 단계가 자연스럽게 연결된다.
- 모듈화 및 디버깅 용이성:
    - 각 단계가 명확히 분리되어 문제 발생 시 어느 단계에서 오류가 발생했는지 쉽게 확인할 수 있다.
    - 복잡한 작업을 작은 단위로 나누어 체계적으로 관리할 수 있다.
      
### Runnable의 표준 메소드
- 모든 Runnable이 구현하는 공통 메소드
    - **`invoke(input, config:RunnableConfig)->output`**: 단일 입력을 처리하여 결과를 반환.
    - **`batch(input:list, config:RunnableConfig|list[RunnableConfig]) -> list[Output]`**: 여러 입력 데이터들을 한 번에 처리.
    - **`stream(input, config:RunnableConfig) -> Iterator[Output]`**: 입력에 대해 스트리밍 방식으로 응답을 반환.
    - **`assign(**kwargs)`**:
      -  앞 Runnable의 출력 결과에 새로운 key–value 쌍의 Field 추가(assign) 하여 다음 Runnable로 전달.
      -  값으로는 Runnable 객체(LCEL체인등)나 고정 값(리터럴) 모두 가능하며, 각 항목은 실행 시 평가되어 기존 출력에 병합한다.
      -  주로 앞 단계의 출력에 부가 정보(field)를 추가하고자 할 때 사용한다. 특히 `RunnablePassthrough`와 결합해, 입력을 그대로 넘기면서 특정 field만 추가할 때 자주 사용


### Runnable의 주요 구현체(하위 클래스)

- 다음 클래스들은 기능을 제공하는 것이 아니라 **chain 구조를 다양하게 구성** 할 수 있도록 도와주는 **Runnable** 타입의 클래스들이다.

- **`RunnableSequence`**
    - 여러 `Runnable`을 순차적으로 연결하여 실행하는 구성이다.
    - 각 단계의 출력이 다음 단계의 입력으로 전달된다.
    - 보통은 LCEL 문법을 사용해서 정의한다.
      - LCEL을 사용하여 체인을 구성할 경우 자동으로 `RunnableSequence`로 변환된다.


In [9]:
from langchain_core.runnables import RunnableSequence

# chain = RunnableSequence(prompt, model, parser)
chain = prompt | model | parser # 위와같음
chain.invoke({"item":"물", "count":2})

'1) **물빛바다**  \n2) **청수온천**'


- **`RunnableLambda`**
    - Lambda 표현식의 함수를 `Runnable`로 변환할 때 사용한다.    
    - 일반함수도 `RunnableLambda`로 변환할 수 있다. 단 일반 함수는 변환 없이 chain에 포함 시킬 수 있기 때문에 굳이 변환할 필요가 없다.
    - Runnable로 만들 함수 구문
        - parameter: 입력 값 1개선언.
        - return: 다음 chain에 전달할 값의 형식


In [12]:
from langchain_core.runnables import RunnableLambda

# RunnableLambda(함수)
# 함수 - 파라미터(1개 -> 앞 체인으로 부터 받을 값에 맞춘다.)
#      - 리턴 값 -> 다음 체인의 입력 타입에 맞게 반환.
c1 = RunnableLambda(lambda input_data:f"{input_data}에 대해서 한 문장으로 설명해줘.")
# type(c1)
c1.invoke("LLM 모델")

'LLM 모델에 대해서 한 문장으로 설명해줘.'

In [13]:
chain = c1 | model | parser
chain.invoke("LLM 모델")  # chain 호출 시 에는 첫번째 컴퓨넌트에 전달할 값을 넣어서 호출 (지금은 c1)

'LLM(대규모 언어 모델)은 대량의 텍스트를 학습해 다음에 올 단어를 예측하는 방식으로 자연어를 이해하고 생성하는 인공지능 모델입니다.'

In [15]:
prompt = ChatPromptTemplate(
    messages = [
        ("system", "모든 응답은 100글자 이내로 작성해줘."),
        ("user", "{query}")
    ]
)
model = ChatOpenAI(model="gpt-5.4-mini")
parser = StrOutputParser()

# Chain구성. chain 응답 : LLM 응답내용, 글자수
chain = prompt | model | parser | RunnableLambda(lambda x : (x, len(x)))


In [16]:
res = chain.invoke("AI에 대해서 설명해줘.")
print(res)

('AI는 사람처럼 학습·판단·생성하는 인공지능입니다. 데이터로 패턴을 익혀 일을 돕습니다.', 49)


In [ ]:
def get_value_len(value:str):
    return value, len(value)

# 일반함수(callable)를 chain의 구성으로 포함시킬 수 있다. -> 내부적으로 Runnable로 변환되서 들어간다.
chain2 = prompt | model | parser | get_value_len    #RunnableLambda(get_value_len)
chain2.invoke("크리스마스")

('메리 크리스마스! 🎄', 11)


-  **`RunnablePassthrough`**
    - 입력 데이터를 가공하지 않고 그대로 다음 단계로 전달하는 `Runnable`이다.
      - 앞 Runnable으로 부터 전달 받은 **입력 값을 다음 Runnable로 그대로 전달**한다.
           - `RunnablePassthrough()`
      - 입력받은 값에 **Field를 추가**해서 전달할 경우 `assign()` 메소드를 사용한다.
           - `RunnablePassthrough.assign(new_key1="new_value1", new_key2="new_value2", ..)`


In [20]:
from langchain_core.runnables import RunnablePassthrough

rp = RunnablePassthrough()  # 단순히 받은 값을 다음으로 통과시킨다.

result = rp.invoke("안녕하세요")
result = rp.invoke([1, 2, 3, 4, 5])
result = rp.invoke({"a":10, "b":20})
print(result)

{'a': 10, 'b': 20}


In [21]:
# 입력받은 값(딕셔너리)에 item을 추가해서 다음으로 전달.
r1 = RunnableLambda(lambda x: "부천시 오정구 고강동")
r2 = RunnableLambda(lambda x: "010-1111-2222")

# assign(key=Runnable, ...)
## 입력받은 딕셔너리에 address와 tel_no key를 추가. value는 Runnable을 호출해서 반환된 값을 설정.
rp2 = RunnablePassthrough.assign(
    address=r1,
    tel_no=r2
)
result = rp2.invoke({"name":"홍길동"})
result

{'name': '홍길동', 'address': '부천시 오정구 고강동', 'tel_no': '010-1111-2222'}


- **`RunnableParallel`**
    - 여러 `Runnable`을 병렬로 실행한 후, 결과를 결합하여 다음 단계로 전달한다.
    - 
        ```python
        RunnableParallel(
            {
                "key1":Runnable1, 
                "key2":Runnable2,
                "key3":Runnable3, ...
            }
        )
        ```
    - 각 Runnable의 실행결과를 Value로 Dictionary를 생성해서 반환한다.
    - LCEL로 정의할 때는 Chain에 dictionary로 정의한다.



In [ ]:
from langchain_core.runnables import RunnableParallel

r1 = RunnableLambda(lambda x : x + 10)
r2 = RunnableLambda(lambda x : x - 10)
r3 = RunnableLambda(lambda x : x * 10)
r4 = RunnableLambda(lambda x : x / 10)

# c = r1 | r2 | r3 | r4

parallel = RunnableParallel(
    {
        "value1" : r1,
        "value2" : r2,
        "value3" : r3,
        "value4" : r4,
        "org_value": RunnablePassthrough()  # 입력 받은 값을 그대로 다음으로 넘겨야 할 경우.
    }
)

result = parallel.invoke(200)
result

{'value1': 210,
 'value2': 190,
 'value3': 2000,
 'value4': 20.0,
 'org_value': 200}

In [26]:
c = RunnablePassthrough() | {
        "value1" : r1,
        "value2" : r2,
        "value3" : r3,
        "value4" : r4,
        "org_value": RunnablePassthrough()  
}

c.invoke(2000)

{'value1': 2010,
 'value2': 1990,
 'value3': 20000,
 'value4': 200.0,
 'org_value': 2000}

### LCEL Chain 예제

In [ ]:
##################################################
# TODO 1
# 음식 이름을 입력하면 그 음식의 레시피를 llm이 출력하는 Chain을 LCEL 을 이용해서 구성한다.
# 입력 : 음식 이름 - recipe_chain.invoke({"food":"김치찌게"})
# 출력 : 음식의 레시피 - 김치찌게 레시피. 

# chain구성: prompt_template -> model(gpt-5-mini) -> StrOutputParser

In [31]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template(
    """ 
    당신은 요리 전문가 입니다.
    사용자가 입력한 음식의 레시피를 자세하게 알려주세요.

    음식:{food}
    """
)

model = ChatOpenAI(
    model='gpt-5.4-mini',
    temperature=0.7
)

parser = StrOutputParser()

recipe_chain = prompt | model | parser

result = recipe_chain.invoke({"food":"김치찌개"})

print(result)

물론입니다. **김치찌개**를 집에서 맛있게 끓일 수 있도록 자세히 알려드릴게요.

---

## 김치찌개 레시피
**인분:** 2~3인분  
**조리시간:** 약 30분

### 재료
- 신김치 1컵~1.5컵
- 돼지고기(목살 또는 앞다리살) 150g
- 두부 1/2모
- 양파 1/2개
- 대파 1대
- 다진 마늘 1큰술
- 고춧가루 1큰술
- 김치 국물 1/2컵
- 물 또는 멸치육수 3컵
- 식용유 1큰술
- 설탕 1작은술
- 국간장 1큰술
- 소금 약간
- 후추 약간

---

## 만드는 방법

### 1) 재료 준비
- **신김치**는 먹기 좋게 잘라주세요.
- **돼지고기**는 한입 크기로 썰어주세요.
- **두부**는 먹기 좋게 깍둑썰기.
- **양파**는 채 썰고, **대파**는 어슷썰기 해주세요.

### 2) 김치와 돼지고기 볶기
- 냄비에 식용유를 두르고 중불로 달굽니다.
- 돼지고기를 먼저 넣고 살짝 볶아 겉면이 익으면,
- 신김치를 넣고 2~3분 정도 함께 볶아주세요.
- 김치가 투명해지고 향이 올라오면 맛이 더 좋아집니다.

### 3) 양념 넣기
- 다진 마늘 1큰술
- 고춧가루 1큰술
- 설탕 1작은술
- 국간장 1큰술
을 넣고 잘 섞어 볶아주세요.

### 4) 끓이기
- 물 또는 멸치육수 3컵과 김치 국물 1/2컵을 넣습니다.
- 센 불에서 끓이다가 끓기 시작하면 중불로 줄여 10~15분 끓여주세요.

### 5) 마무리
- 양파와 두부를 넣고 5분 정도 더 끓입니다.
- 마지막에 대파를 넣고 한소끔 더 끓인 뒤,
- 부족한 간은 소금이나 국간장으로 맞춥니다.
- 후추를 약간 넣으면 풍미가 좋아집니다.

---

## 맛있게 만드는 팁
- **김치는 꼭 신김치**를 사용해야 감칠맛이 살아납니다.
- 돼지고기는 **목살이나 앞다리살**이 가장 잘 어울립니다.
- 더 깊은 맛을 원하면 **멸치육수**나 **쌀뜨물**을 사용하세요.
- 찌개를 한 번 끓인 뒤 **5분 정도 잠시 두었다가 다시 데우면** 맛이 더 잘 배어듭니다.

--

In [32]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

system_prompt = """ 
<instruction>
당신은 요리전문 AI Assistant 입니다.
요청받은 음식의 레시피를 자세하고 쉽게 작성해주세요.
출력 방법은 아래 output_format을 참고해서 응답해주세요.
</instruction>

<output_format>
- Markdown 형식으로 답변을 작성합니다.
- 응답 내용에는 다음 항목들을 포함합니다.
    - 요리 이름
    - 요리 기본정보
        - 난이도(★*10개가 만점)
        - 조리시간
        - 인분
    - 요리에 필요한 재료
    - 요리 방법
    - 팁
</output_format>
"""

parser = StrOutputParser()
model = ChatOpenAI(model='gpt-5.4-mini')
prompt = ChatPromptTemplate(
    messages = [
        ("system", system_prompt),
        ("user","{food}의 레시피를 작성해주세요.")
    ]
)

In [ ]:
recipe_chain = prompt | model | parser

res = recipe_chain.invoke({"food":"김치찌개"})



# 김치찌개 레시피

## 요리 기본정보
- **난이도**: ★★★☆☆☆☆☆☆☆  
- **조리시간**: 약 30~40분  
- **인분**: 2~3인분  

## 요리에 필요한 재료
- 잘 익은 김치 1컵~1.5컵
- 돼지고기(앞다리살 또는 삼겹살) 150g
- 두부 1/2모
- 양파 1/2개
- 대파 1대
- 다진 마늘 1큰술
- 고춧가루 1큰술
- 김치 국물 1/2컵
- 물 또는 멸치육수 2~3컵
- 식용유 1큰술
- 설탕 1작은술
- 소금 약간
- 후추 약간

## 요리 방법
1. **재료 손질하기**  
   김치는 먹기 좋은 크기로 썰고, 돼지고기는 한입 크기로 준비합니다. 두부는 큼직하게 썰고, 양파는 채 썰고, 대파는 어슷하게 썹니다.

2. **돼지고기 볶기**  
   냄비에 식용유를 두르고 돼지고기를 넣어 중불에서 볶습니다. 고기의 겉면이 익으면 다진 마늘을 넣고 함께 볶아 향을 냅니다.

3. **김치 넣고 볶기**  
   김치를 넣고 고춧가루, 설탕을 넣은 뒤 3~5분 정도 볶습니다. 김치가 충분히 볶아져야 김치찌개의 감칠맛이 살아납니다.

4. **국물 붓고 끓이기**  
   김치 국물과 물 또는 멸치육수를 넣고 끓입니다. 끓기 시작하면 중약불로 줄여 15분 정도 더 끓입니다.

5. **재료 마무리 넣기**  
   양파와 두부를 넣고 5분 정도 더 끓입니다. 마지막에 대파를 넣고 소금, 후추로 간을 맞춥니다.

6. **완성하기**  
   모든 재료가 잘 어우러지면 불을 끄고 그릇에 담아 따뜻하게 먹습니다.

## 팁
- **김치는 꼭 잘 익은 김치**를 사용해야 깊은 맛이 납니다.
- 돼지고기 대신 **참치, 스팸, 꽁치**를 넣어도 맛있습니다.
- 더 진한 맛을 원하면 **멸치육수**를 사용하세요.
- 찌개는 한 번 식혔다가 다시 끓이면 맛이 더 깊어집니다.


In [ ]:
from IPython.display import Markdown
# print(res)
Markdown(res)

# 김치찌개 레시피

## 요리 기본정보
- **난이도**: ★★★☆☆☆☆☆☆☆  
- **조리시간**: 약 30~40분  
- **인분**: 2~3인분  

## 요리에 필요한 재료
- 잘 익은 김치 1컵~1.5컵
- 돼지고기(앞다리살 또는 삼겹살) 150g
- 두부 1/2모
- 양파 1/2개
- 대파 1대
- 다진 마늘 1큰술
- 고춧가루 1큰술
- 김치 국물 1/2컵
- 물 또는 멸치육수 2~3컵
- 식용유 1큰술
- 설탕 1작은술
- 소금 약간
- 후추 약간

## 요리 방법
1. **재료 손질하기**  
   김치는 먹기 좋은 크기로 썰고, 돼지고기는 한입 크기로 준비합니다. 두부는 큼직하게 썰고, 양파는 채 썰고, 대파는 어슷하게 썹니다.

2. **돼지고기 볶기**  
   냄비에 식용유를 두르고 돼지고기를 넣어 중불에서 볶습니다. 고기의 겉면이 익으면 다진 마늘을 넣고 함께 볶아 향을 냅니다.

3. **김치 넣고 볶기**  
   김치를 넣고 고춧가루, 설탕을 넣은 뒤 3~5분 정도 볶습니다. 김치가 충분히 볶아져야 김치찌개의 감칠맛이 살아납니다.

4. **국물 붓고 끓이기**  
   김치 국물과 물 또는 멸치육수를 넣고 끓입니다. 끓기 시작하면 중약불로 줄여 15분 정도 더 끓입니다.

5. **재료 마무리 넣기**  
   양파와 두부를 넣고 5분 정도 더 끓입니다. 마지막에 대파를 넣고 소금, 후추로 간을 맞춥니다.

6. **완성하기**  
   모든 재료가 잘 어우러지면 불을 끄고 그릇에 담아 따뜻하게 먹습니다.

## 팁
- **김치는 꼭 잘 익은 김치**를 사용해야 깊은 맛이 납니다.
- 돼지고기 대신 **참치, 스팸, 꽁치**를 넣어도 맛있습니다.
- 더 진한 맛을 원하면 **멸치육수**를 사용하세요.
- 찌개는 한 번 식혔다가 다시 끓이면 맛이 더 깊어집니다.

In [36]:
res2 = recipe_chain.invoke({"food":"에스카르고"})
Markdown(res2)

# 에스카르고 레시피

## 요리 기본정보
- **난이도**: ★★★★☆☆☆☆☆☆
- **조리시간**: 약 35분
- **인분**: 2~3인분

## 요리에 필요한 재료
- 에스카르고(달팽이) 살 12개
- 무염버터 100g
- 마늘 4~5쪽
- 파슬리 2큰술(잘게 다진 것)
- 샬롯 1개(또는 양파 1/4개, 잘게 다짐)
- 화이트와인 2큰술
- 소금 약간
- 후추 약간
- 레몬즙 1작은술
- 빵가루 1큰술(선택)
- 에스카르고 껍데기 또는 작은 오븐용 그릇
- 곁들임용 바게트 약간

## 요리 방법
1. **재료 손질하기**  
   마늘과 샬롯은 아주 곱게 다지고, 파슬리는 깨끗이 씻어 물기를 제거한 뒤 잘게 썹니다.

2. **허브버터 만들기**  
   볼에 무염버터를 넣고 실온에서 부드럽게 풀어줍니다.  
   여기에 다진 마늘, 샬롯, 파슬리, 화이트와인, 레몬즙, 소금, 후추를 넣고 잘 섞습니다.  
   원하면 빵가루를 넣어 약간 더 고소하고 농도 있게 만들어도 좋습니다.

3. **에스카르고 준비하기**  
   에스카르고 살은 흐르는 물에 가볍게 헹군 뒤 물기를 제거합니다.  
   이미 익혀진 제품이라면 바로 사용하고, 생 달팽이라면 반드시 안전하게 손질·가열한 제품만 사용해야 합니다.

4. **껍데기 또는 용기에 담기**  
   에스카르고 껍데기 안쪽에 허브버터를 조금 넣고, 그 위에 에스카르고를 1개씩 넣습니다.  
   다시 위를 버터로 채워 입구를 덮어줍니다.  
   껍데기가 없다면 작은 오븐용 그릇에 담아도 됩니다.

5. **오븐에 굽기**  
   200℃로 예열한 오븐에서 8~10분 정도 구워 버터가 보글보글 끓고 향이 올라오게 합니다.  
   마지막 1~2분은 윗불을 강하게 해 겉면을 살짝 노릇하게 만들어도 좋습니다.

6. **마무리 후 서빙하기**  
   뜨거울 때 바로 꺼내 바게트와 함께 담아 냅니다.  
   남은 허브버터는 바게트에 찍어 먹으면 아주 잘 어울립니다.

## 팁
- **에스카르고는 안전한 식용 제품**을 사용하세요. 생달팽이를 직접 조리하는 경우는 위생과 손질이 매우 중요합니다.
- 허브버터는 **미리 만들어 냉장 보관**하면 편리합니다. 필요할 때 바로 꺼내 사용하면 됩니다.
- **마늘 향을 더 진하게** 즐기고 싶다면 마늘을 조금 늘려도 좋습니다.
- 바게트 외에도 **토스트, 파스타, 구운 감자**와 곁들이면 색다르게 즐길 수 있습니다.

In [ ]:
##############################################################
#  TODO 2
# 번역할 내용, 번역할 언어 를 입력하면 내용을 그 언어로 번역하는 Chain을 LCEL 을 이용해서 구성한다.
#
## 입력: 번역할 내용, 언어.  translate_chain.invoke({"content":"안녕하세요.", "language":"영어"})
## 출력: "번역할 내용"을 "언어" 로 번역한 결과 - "How are you?".

# chain구성: prompt_template -> model(gpt-5-mini) -> StrOutputParser

In [29]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template(
    """
    다음 내용을 {language}로 번역해주세요.

    내용:
    {content}

    번역 결과만 출력하세요.
    """
)

model = ChatOpenAI(
    model="gpt-5.4-mini",
    temperature=0
)

parser = StrOutputParser()

translate_chain = prompt | model | parser

result = translate_chain.invoke(
 {   "content" : "오늘은 빨리 집에 가고 싶다.",
     "language": "일본어"
 }
)

print(result)

今日は早く家に帰りたい。


In [37]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

prompt = ChatPromptTemplate.from_template("{content}를{language}로 번역해주세요.")

model = ChatOpenAI(model="gpt-5.4-mini")

parser = StrOutputParser()

translate_chain = prompt | model | parser

result = translate_chain.invoke(
 {   "content" : "밥은 먹었니?",
     "language": "중국어"
 }
)

print(result)

“밥은 먹었니?”를 중국어로 하면 보통:

**你吃饭了吗？**  
(pinyin: **Nǐ chī fàn le ma?**)

좀 더 자연스럽게 인사처럼 쓰면 **你吃了吗？**라고도 합니다.


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

system_prompt2 = """ 
<instruction>
당신은 다국어가 가능한 숙련된 번역 AI Assistant 입니다.
<input_data> 항목에 작성된 내용을 참고 해서 **요청된 문서**의 내용을 **요청된 언어**로 번역해 주세요.
내용의 의미를 해치지 않는 범위에서 최대한 읽기 쉽게 작성해 주세요.
</instruction>

<input_data>
- 번역할 내용: {content}
- 번역할 언어: {language}
</input_data>
"""

prompt_trans = ChatPromptTemplate.from_template(
   template=system_prompt2
)
model_trans = ChatOpenAI(model='gpt-5.4-mini')

translate_chain = prompt_trans | model_trans | StrOutputParser()

In [39]:
content = """The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolutions
entirely. Experiments on two machine translation tasks show these models to
be superior in quality while being more parallelizable and requiring significantly
less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.8 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature. We show that the Transformer generalizes well to
other tasks by applying it successfully to English constituency parsing both with
large and limited training data."""

res = translate_chain.invoke({"content":content,"language":"중국어"})

In [40]:
print(res)

主流的序列转换模型基于复杂的循环神经网络或卷积神经网络，这些网络包含编码器和解码器。性能最好的模型还通过注意力机制将编码器和解码器连接起来。我们提出了一种新的简单网络架构——Transformer，它完全基于注意力机制，彻底摒弃了循环和卷积。对两项机器翻译任务的实验表明，这些模型在质量上更优，同时更易于并行化，并且所需训练时间显著更少。我们的模型在 WMT 2014 英语到德语翻译任务上取得了 28.4 的 BLEU 分数，比现有最佳结果（包括集成模型）提高了 2 个多 BLEU。 在 WMT 2014 英语到法语翻译任务上，我们的模型在使用 8 块 GPU 训练 3.5 天后，取得了 41.8 的全新单模型最佳 BLEU 分数；这只是文献中最佳模型训练成本的一小部分。我们还通过将 Transformer 成功应用于英语成分句法分析，分别在大规模和有限训练数据条件下，证明了它对其他任务也具有良好的泛化能力。


In [41]:
from langchain_core.runnables import Runnable

isinstance(translate_chain, Runnable)

True

### Chain과 Chain간의 연결

In [ ]:
from operator import itemgetter

ig = itemgetter("language")

a = {"language":"영어", "name":"홍길동"}
ig(a)   # 자료구조를 넣어주면 생성할 때 지정한 index/key의 값을 조회해서 반환

ig2 = itemgetter(2) # 2번째 값 추출
b = [10, 20, 30, 40, 50, 60]
ig2(b)

30

In [49]:
# 음식 레시피를 원하는 언어로 출력하는 AI Agent가 필요.
## recipe_chain 과 translate_chain을 연결
from langchain_core.runnables import RunnableLambda
from operator import itemgetter
# {}: RunnableParallel
chain = {
    "content":recipe_chain,
    #"language":RunnableLambda(lambda x : x['language'])
    "language":itemgetter("language")   # 위와 같음
} | translate_chain

In [50]:
res = chain.invoke({"food":"해물짜장면", "language":"베트남어"})

In [51]:
print(res)

# Mì Jajang hải sản

## Thông tin cơ bản
- **Độ khó**: ★★★☆☆☆☆☆☆☆  
- **Thời gian nấu**: khoảng 30 phút  
- **Khẩu phần**: 2 phần  

## Nguyên liệu cần chuẩn bị
### Nguyên liệu chính
- 2 phần mì Trung Quốc
- 1/2 con mực
- 6–8 con tôm
- Khoảng 10 con nghêu hoặc vẹm
- 1 củ hành tây
- 2 nắm bắp cải
- 1/3 quả bí xanh Hàn Quốc (애호박)
- 1 củ khoai tây
- 1/2 cây hành boa-rô
- 3 muỗng canh dầu ăn

### Nguyên liệu cho sốt jajang
- 4 muỗng canh tương đen Hàn Quốc (chunjang)
- 2 cốc nước
- 2 muỗng canh tinh bột
- 4 muỗng canh nước (để pha tinh bột)
- 1–2 muỗng canh đường
- 1 muỗng canh nước tương
- 1 muỗng canh dầu hào
- 1 muỗng canh tỏi băm
- Một ít tiêu
- Một ít muối

### Gia vị sơ chế hải sản
- 1 muỗng canh rượu nấu ăn
- Một ít tiêu

## Cách làm
1. **Sơ chế nguyên liệu**  
   Mực bỏ nội tạng, rửa sạch rồi cắt miếng vừa ăn. Tôm bóc vỏ, nghêu hoặc vẹm ngâm nước muối để nhả cát rồi chuẩn bị.  
   Hành tây, bắp cải, bí xanh, khoai tây và hành boa-rô cắt thành miếng vừa ăn.

2. **Ướp sơ hải sản**  


## 함수를 Runnable로 정의하기

### 함수 구현
- **파라미터**
   - 이전 Chain에서 출력한 값을 입력으로 받을 수 있도록 정의한다.
- **리턴값**
   - 다음 Chain으로 입력할 값을 반환하도록 구현한다.

### Runnable 타입으로 만들기
1. LCEL Chain안에 함수를 구성요소로 포함시키면, 그 함수는 자동으로 `Runnable` 로 취급된다.
   - 별도의 래핑이나 추가 처리는 필요하지 않다.
2. `RunnableLambda()` 에 넣어 명시적으로 `Runnable` 타입으로 만든다.
   - Lambda 표현식으로 정의할 경우 `RunnableLambda(lambda 표현식)` 으로 정의해야 한다.
   - 보통 일반함수는 `RunnableLambda`를 사용할 필요 없다.
3. `@chain` decorator를 사용
   - 함수에 `@chain` decorator가 선언되면 그 함수는 `RunnableLambda` 타입이 된다.
   - 이 방식은 LCEL만으로 표현하기 어려운 실행 흐름을 직접 정의해야 할 때 주로 사용된다.
     - LCEL은 순차 실행구조를 따른다. 그래서  제어문을 이용해 그 흐름을 제어할 수가 없다. 
     - 단순한 파이프라인에서는 LCEL만으로도 충분하지만, 다음과 같은 경우에는 한계가 있다.
       - 특정 단계를 조건에 따라 실행하거나 생략해야 하는 경우
       - 동일한 단계를 반복적으로 실행해야 하는 경우
       - 여러 판단 로직에 따라 실행 경로가 달라지는 agent 구조
     - 이처럼 복잡한 업무 흐름을 가지는 agent는 단순한 순차 구조만으로는 원하는 응답 품질을 얻기 어렵다. 결국 실행 흐름 자체를 개발자가 직접 코드로 정의해야 하며, 이러한 경우에 `@chain`을 사용해 chain/agent 함수를 구현한다.
   - 이러한 복잡한 실행 흐름을 보다 구조적으로 정의하기 위해 LangChain에서 추가로 제공하는 것이 **LangGraph**이다.

In [53]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

def plus(n1,n2):
    return n1 + n2

def wrap_plus(x):
    return plus(x[0], x[1])

# chain = RunnablePassthrough() | RunnableLambda(lambda x : plus(x[0], x[1]))
chain = RunnablePassthrough() | wrap_plus
chain.invoke([1,2])

3

In [58]:
# recipe_chain 과 translate_chain을 이용해서 음식레시피를 특정 언어로 반환.
## recipe_chain의 결과와 traslate_chain의 결과 둘을 모두 반환.

from langchain_core.runnables import chain

# RunnableLambda(multi_language_recipe_chain)

@chain
def multi_language_recipe_chain(input_data: dict) -> dict[str, str]:
    food: str = input_data['food']                # 음식 이름
    language: str = input_data['language']   # 레시피 언어
    is_korean: bool = input_data['is_korean'] # 한국어 레시피 필요 여부.

    korean_recipe = recipe_chain.invoke({"food":food})

    result = translate_chain.invoke({"content": korean_recipe, "language": language})

    final_result = {"recipe":result}

    if is_korean: # 한국어 레시피도 요청
        final_result["korean_recipe"] = korean_recipe
    
    return final_result

In [59]:
res = multi_language_recipe_chain.invoke(
    {"food":"돈까스", "language":"일본어", "is_korean":True}
)

In [71]:
print(res['recipe'])

# トンカツレシピ

## 基本情報
- **難易度：** ★★★☆☆☆☆☆☆☆  
- **調理時間：** 約30分  
- **人数：** 2人分  

## 材料
- 豚ロースまたはヒレ肉 2枚
- 塩 少々
- こしょう 少々
- 小麦粉 1/2カップ
- 卵 2個
- パン粉 2カップ
- サラダ油 適量

### ソースの材料
- とんかつソースまたはウスターソース 大さじ4
- ケチャップ 大さじ2
- 水 大さじ2
- 砂糖 小さじ1

## 作り方
1. **肉の下準備**  
   豚肉は包丁の背や肉たたきで軽くたたき、厚さを均一にします。  
   厚すぎると火が通るまで時間がかかり、薄すぎると硬くなりやすいので、ほどよくのばしてください。

2. **下味をつける**  
   下準備した肉の両面に塩とこしょうをまんべんなくふって下味をつけます。

3. **衣の準備**  
   - 小麦粉  
   - 溶き卵  
   - パン粉  
   をそれぞれ別の皿に入れて準備します。

4. **衣をつける**  
   まず肉に薄く小麦粉をまぶし、次に溶き卵にくぐらせ、最後にパン粉をしっかり押さえるようにつけます。  
   パン粉がよく付くように、手で軽く押さえてください。

5. **揚げる**  
   フライパンにサラダ油を十分に入れ、170〜180℃程度に熱したらトンカツを入れます。  
   両面をこんがり揚げ、片面あたり約3〜4分ずつ火を通します。  
   強火すぎず、中火で中までしっかり火を通すのがポイントです。

6. **油を切る**  
   揚げたトンカツはキッチンペーパーや網の上に置いて油を切ります。

7. **ソースを作る**  
   鍋にソースの材料を入れ、弱火で軽く煮てとろみをつけます。  
   お好みに合わせて水の量を調整し、濃度を整えてください。

8. **仕上げる**  
   トンカツを食べやすい大きさに切って皿に盛り、ソースをかけるか添えれば完成です。

## ポイント
- **肉を事前に10〜15分ほど室温に戻すと**、より均一に火が通ります。
- **パン粉をもう一度しっかり押さえると**、揚げるときに衣がはがれにくくなります。
- **油の温度が低すぎると**油を吸いやすく、高すぎる

# Cache

- 응답 결과를 저장해서 같은 질문이 들어오면 LLM에 요청하지 않고 저장된 결과를 보여주도록 한다.
    - 처리속도와 비용을 절감할 수 있다.
    - 특히 chatbot같이 비슷한 질문을 하는 경우 유용하다.
- 저장 방식은 `메모리`, `sqlite` 등 다양한 방식을 지원한다.
  
    ```python
    set_llm_cache(Cache객체)
    ```

In [76]:
from langchain_core.globals import set_llm_cache
from langchain_community.cache import InMemoryCache, SQLiteCache

# Cache 설정은 한번만 하면 된다.
# set_llm_cache(InMemoryCache())
set_llm_cache(SQLiteCache("cache.sqlite"))  # cache를 저장할 파일 경로


In [77]:
res = multi_language_recipe_chain.invoke(
    {"food":"돈까스", "language":"중국어", "is_korean":True}
)

In [78]:
print(res['korean_recipe'])

# 돈까스 레시피

## 요리 기본정보
- **난이도:** ★★★★☆☆☆☆☆☆
- **조리시간:** 30분
- **인분:** 2인분

## 요리에 필요한 재료
### 주재료
- 돼지고기 등심 2장
- 소금 약간
- 후추 약간
- 밀가루 1/2컵
- 달걀 2개
- 빵가루 1~2컵
- 식용유 적당량

### 곁들임
- 양배추 채썬 것
- 돈까스 소스
- 밥

## 요리 방법
1. **돼지고기 손질하기**  
   돼지고기 등심은 키친타월로 핏물을 닦아낸 뒤, 고기망치나 칼등으로 두드려 두께를 고르게 펴줍니다.  
   너무 두꺼우면 익는 시간이 길어지고 질길 수 있으니 적당히 얇게 펴주세요.

2. **간하기**  
   앞뒤로 소금과 후추를 살짝 뿌려 밑간을 합니다.  
   너무 많이 뿌리면 소스와 함께 먹을 때 짤 수 있으니 가볍게만 해주세요.

3. **튀김옷 준비하기**  
   각각 접시에 밀가루, 풀어놓은 달걀, 빵가루를 준비합니다.  
   고기 순서대로 **밀가루 → 달걀물 → 빵가루** 순서로 입혀줍니다.  
   빵가루를 누르듯이 잘 붙여야 튀길 때 떨어지지 않습니다.

4. **튀기기**  
   팬에 식용유를 넉넉히 붓고 170~180℃ 정도로 예열합니다.  
   빵가루를 살짝 넣었을 때 바로 올라오면 적당한 온도입니다.  
   돈까스를 넣고 앞뒤로 노릇노릇하게 튀겨줍니다.  
   보통 한 면당 2~3분 정도, 전체 5~7분 정도가 걸립니다.

5. **기름 빼기**  
   튀긴 돈까스는 키친타월이나 망 위에 올려 기름을 빼줍니다.  
   바로 자르면 육즙이 빠질 수 있으니 1~2분 정도 잠시 두는 것이 좋습니다.

6. **마무리하기**  
   먹기 좋게 썰어 접시에 담고, 채썬 양배추와 밥을 곁들입니다.  
   돈까스 소스를 위에 뿌리거나 따로 담아 내면 완성입니다.

## 팁
- **고기를 두드릴 때** 너무 세게 하면 찢어질 수 있으니 적당히 펴주세요.
- **빵가루는 손으로 꾹 눌러** 붙이면 튀김옷이 더 잘 유지됩니다

In [79]:
print(res['recipe'])

# 日式炸猪排食谱

## 烹饪基本信息
- **难度：** ★★★★☆☆☆☆☆☆
- **烹饪时间：** 30分钟
- **份量：** 2人份

## 所需食材
### 主料
- 猪里脊肉 2片
- 盐 少许
- 黑胡椒 少许
- 面粉 1/2杯
- 鸡蛋 2个
- 面包糠 1～2杯
- 食用油 适量

### 配菜
- 切丝卷心菜
- 炸猪排酱
- 米饭

## 制作方法
1. **处理猪肉**  
   用厨房纸巾擦去猪里脊肉表面的血水后，用肉锤或刀背轻轻敲打，使肉片厚薄均匀。  
   如果太厚，熟得会比较慢，而且口感可能偏柴，所以请适当拍薄。

2. **调味**  
   在猪肉两面轻撒少许盐和黑胡椒进行基础调味。  
   不要放太多，否则配上酱汁后可能会偏咸，轻轻调味即可。

3. **准备裹粉材料**  
   分别在盘子里准备好面粉、打散的鸡蛋和面包糠。  
   按照 **面粉 → 蛋液 → 面包糠** 的顺序给猪肉裹上。  
   面包糠要用手轻轻按压，使其牢固附着，这样炸的时候才不容易脱落。

4. **油炸**  
   在平底锅中倒入足量食用油，加热至约 170～180℃。  
   当放入少许面包糠后立刻浮起，就表示油温合适。  
   放入炸猪排，两面炸至金黄。  
   通常每面约需 2～3 分钟，全程约 5～7 分钟。

5. **沥油**  
   炸好的炸猪排放在厨房纸巾或网架上沥油。  
   不要马上切开，稍微静置 1～2 分钟，让肉汁稳定下来。

6. **完成摆盘**  
   将炸猪排切成方便食用的大小，盛入盘中，搭配切丝卷心菜和米饭。  
   最后淋上炸猪排酱，或者将酱汁另外装盘，即可完成。

## 小贴士
- **敲打肉片时** 不要太用力，以免把肉拍破。
- **面包糠用手按压** 后，裹衣会更牢固。
- **油温** 太低会吸油，太高则容易外焦里生。
- 如果想要更酥脆，可以在裹面包糠时 **再轻轻按压一次**，让外层更厚实。
- 也可以使用空气炸锅，但用平底锅油炸的炸猪排通常更酥脆。
